<a href="https://colab.research.google.com/github/ASAND5/lazy_projects/blob/main/RAG_from_Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
!pip install -q pypdf

import glob
import os
from pypdf import PdfReader
import gc
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

In [22]:
print("Is CUDA (GPU) available?:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))
else:
    print("WARNING: You are still on a CPU runtime. Please change your runtime to GPU in the menu: Runtime -> Change runtime type -> T4 GPU.")

Is CUDA (GPU) available?: True
GPU Device Name: Tesla T4


In [25]:
folder_path = "/content/drive/MyDrive/RAG/"

# List of specific documents you want to load
specific_pdfs = [
    "/content/drive/MyDrive/RAG/Banality_of_Evil_in_Our_July_Uprising.pdf",
    "/content/drive/MyDrive/RAG/Date ESR.pdf",
    "/content/drive/MyDrive/RAG/Date IJAMRS.pdf",
    "/content/drive/MyDrive/RAG/ForBigganKothon.pdf",
    "/content/drive/MyDrive/RAG/The More I Try, The More The World Wants Me To Trip Down Again.pdf"
]

documents = []

# Load the specific PDFs first
for pdf_path in specific_pdfs:
    if os.path.exists(pdf_path):
        reader = PdfReader(pdf_path)
        pdf_text = "\n".join(page.extract_text() or "" for page in reader.pages)
        documents.append(pdf_text)
        print(f"Loaded specified report: {os.path.basename(pdf_path)}")
    else:
        print(f"Warning: Could not find {os.path.basename(pdf_path)}")

print(f"\nLoaded {len(documents)} total documents from your specified list")

Loaded specified report: Banality_of_Evil_in_Our_July_Uprising.pdf
Loaded specified report: Date ESR.pdf
Loaded specified report: Date IJAMRS.pdf
Loaded specified report: ForBigganKothon.pdf
Loaded specified report: The More I Try, The More The World Wants Me To Trip Down Again.pdf

Loaded 5 total documents from your specified list


In [26]:
def chunk_texts(docs, chunk_size=20, overlap=5):
    chunks = []
    for doc_id, text in enumerate(docs):
        words = text.split()
        start = 0
        while start < len(words):
            end = start + chunk_size
            chunk_str = " ".join(words[start:end])
            chunks.append({"doc_id": doc_id, "text": chunk_str})
            start += max(1, chunk_size - overlap)
    return chunks

# Ensure documents is defined; if not, remind user to run the document loading cell
try:
    chunks = chunk_texts(documents, chunk_size=18, overlap=4)
    print(f"Created {len(chunks)} chunks")
except NameError as e:
    print("Error: 'documents' variable is not defined in memory.")
    print("Please run the preceding cell (cell n88kmoCv6DdT) to load your PDFs first, then run this cell again!")

Created 1552 chunks


In [27]:
# Free up any unused cached memory on the GPU first
gc.collect()
torch.cuda.empty_cache()

# Define the instruction formatting function for E5 models
def format_query(query: str) -> str:
    return f"query: {query}"

# Set device to 'cpu' for embeddings to safely bypass GPU memory limits,
# or keep 'cuda' but use a much smaller batch size if you want to try GPU.
# CPU is highly recommended here since Aion-1.0 is already occupying most GPU VRAM.
embed_device = "cpu"
embed_model = SentenceTransformer("intfloat/multilingual-e5-large", device=embed_device)

# E5 models expect document passages to be prefixed with 'passage: '
chunk_texts_only = [f"passage: {c['text']}" for c in chunks]

# Generate embeddings for the chunks (using CPU with batch_size 128 is safe and stable)
chunk_embeddings = embed_model.encode(
    chunk_texts_only,
    batch_size=128,
    show_progress_bar=True,
    normalize_embeddings=True
)

def retrieve(query: str, top_k: int = 2):
    # Format the query with the E5 query prefix
    formatted_query = format_query(query)
    # Compute query vector (on CPU to match chunk_embeddings device)
    query_vec = embed_model.encode([formatted_query], normalize_embeddings=True)[0]

    # Calculate cosine similarity using the dot product of normalized vectors
    scores = np.dot(chunk_embeddings, query_vec)
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [(chunks[i]["text"], float(scores[i])) for i in top_indices]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

In [28]:
# Load the specified model and tokenizer
model_id = "microsoft/Aion-1.0-Instruct-2.3B"
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True
)

def generate_answer(query, top_k=2):
    # Retrieve relevant contexts using our defined retrieve function
    retrieved = retrieve(query, top_k=top_k)
    context_block = "\n".join([f"- {txt}" for txt, _ in retrieved])

    # Structure the message history with a system instruction and user prompt
    messages = [
        {
            "role": "system",
            "content": "Answer the user's question using ONLY the provided context. If the answer is not in the context, say you don't know."
        },
        {
            "role": "user",
            "content": f"Context:\n{context_block}\n\nQuestion: {query}"
        }
    ]

    # Format standard chat template for Aion-1.0
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Generate output tokens
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False
        )

    # Decode only the generated response portion
    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
    return answer, retrieved


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

In [35]:
# Prompt the user to enter their question via an interactive input box
user_question = input("Please enter your question for the RAG system: ")

if user_question.strip():
    # Generate the answer and retrieve source chunks
    rag_answer, source_chunks = generate_answer(user_question, top_k=3)

    print(f"\nQuestion: {user_question}\n")
    print("Answer:")
    print(rag_answer)
    print("\nRetrieved Reference Chunks:")
    for idx, (text, score) in enumerate(source_chunks, 1):
        print(f"  [{idx}] Similarity Score: {score:.3f}")
        print(f"      Text: {text}\n")
else:
    print("No question was entered.")

Please enter your question for the RAG system: Who was eichman?

Question: Who was eichman?

Answer:
Adolf Eichmann was a former Nazi Schutzstaffel (SS) army officer, specifically a facilitator of Nazi atrocities during World War II. He was indicted for crimes against humanity, including killing people in extermination camps, preventing births, and other atrocities.

Retrieved Reference Chunks:
  [1] Similarity Score: 0.787
      Text: 2. Eichmann was indicted on 12 charges, including killing human beings in ex‐ termination camps, preventing births, forced

  [2] Similarity Score: 0.782
      Text: rare opportunity to observe Adolf Eichmann, a facilitator of Nazi atrocities in world war 2. Eichmann was indicted

  [3] Similarity Score: 0.778
      Text: of a former Nazi Schutzstaffel (SS) army ofϐicer, Adolf Eichmann. So, Arendt covered the trial for The New

